In [1]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader, Subset
import numpy as np
from tqdm import tqdm
import tarfile
import urllib.request
from transformers import AutoImageProcessor, AutoModel
import requests

In [7]:
# --- 1. ROZWIĄZANIE KONFLIKTÓW SYSTEMOWYCH ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. KONFIGURACJA REPOZYTORIUM ---
REPO_NAME = "ssl-data-curation"
REPO_URL = "https://github.com/facebookresearch/ssl-data-curation.git"


#
def setup_repo():
    if not os.path.exists(REPO_NAME):
        print(f"--- Klonowanie repozytorium {REPO_NAME}... ---")
        subprocess.run(["git", "clone", REPO_URL], check=True)

    # Dodanie katalogów do sys.path, aby Python widział folder 'src'
    repo_path = os.path.abspath(REPO_NAME)
    src_path = os.path.join(repo_path, "src")

    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    print("--- Ścieżki repozytorium skonfigurowane ---")


#
setup_repo()
# # --- 3. IMPORTY Z REPOZYTORIUM (PO USTAWIENIU ŚCIEŻEK) ---
try:
    from src.clusters import HierarchicalCluster
    from src import hierarchical_kmeans_gpu as hkmg
    from src import hierarchical_sampling as hs

    print("--- Moduły Facebook Research załadowane pomyślnie ---")
except ImportError as e:
    print(f"Błąd importu: {e}")
    sys.exit(1)
#
# # --- 4. PRZYGOTOWANIE DANYCH (IMAGENETTE) ---
DATA_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
DATA_DIR = "imagenette2-320"


#
#
def prepare_data():
    if not os.path.exists(DATA_DIR):
        print("--- Pobieranie zbioru Imagenette (320px)... ---")
        if not os.path.exists("imagenette.tgz"):
          urllib.request.urlretrieve(DATA_URL, "imagenette.tgz")
        with tarfile.open("imagenette.tgz", "r:gz") as tar:
            tar.extractall(path=DATA_DIR)


# --- 5. GŁÓWNA FUNKCJA TRENINGOWA ---
def train_model(model, train_loader, test_loader, device, title, epochs=5):
    print(f"\n[TRENING] Start: {title}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    skipped_steps_counter = 0
    scaler = torch.amp.GradScaler("cuda")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f"Epoka {epoch + 1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none = True)
            with torch.amp.autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()

            #makeing a step and checking wheter scaler made fp16 overflow
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale()
            if old_scale != new_scale:
                skipped_steps_counter+=1
                print(f"Step skipped, scale reduced from {old_scale} -> {new_scale} (batch was futile)")
            running_loss += loss.item()

    # Ewaluacja
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"[WYNIK] {title} Accuracy: {accuracy:.2f}%")
    return accuracy


def get_resnet50_embedings(dataset):
    extractor = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).to(device)
    extractor.fc = nn.Identity()
    extractor.eval()
    
    all_features = []
    with torch.inference_mode(), torch.amp.autocast("cuda"):
        feature_loader = DataLoader(dataset, batch_size=128, shuffle=False)
        for imgs, _ in tqdm(feature_loader, desc="Ekstrakcja cech"):
            feat = extractor(imgs.to(device))
            all_features.append(feat.cpu())            
    
    return torch.cat(all_features).to(device)

class DinoTransform:
    def __init__(self, processor= AutoImageProcessor.from_pretrained('facebook/dinov2-large') ):
        self.processor = processor
    def __call__(self, img):
        return self.processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)

def get_dino_embedings(dataset):
    #dataset need proper tronsform (DinoTransform)
    extr_loader = DataLoader(dataset,
                             batch_size=64,
                             shuffle=False,
                             num_workers=0) #można 2 w colabie albo na linuxie

    all_embedings = []
    extractor = AutoModel.from_pretrained('facebook/dinov2-large')
    extractor.eval()
    extractor.to(device)
    with torch.inference_mode(), torch.amp.autocast("cuda"):
        for imgs,_ in tqdm(extr_loader, desc="Wyciągnie embedingów: "):
            imgs = imgs.to(device)
            outputs = extractor(imgs)
            embedings = outputs.last_hidden_state[:,0,:]
            all_embedings.append(embedings.cpu())

    return torch.cat(all_embedings).to(device)

--- Ścieżki repozytorium skonfigurowane ---
--- Moduły Facebook Research załadowane pomyślnie ---


In [6]:
# --- 6. URUCHOMIENIE EKSPERYMENTU ---
def main():
    prepare_data()
    print(f"Praca na urządzeniu: {device}")

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    real_data_dir = os.path.join(DATA_DIR,"imagenette2-320")
    full_train_dataset = datasets.ImageFolder(os.path.join(real_data_dir, 'train'), transform=transform)
    test_dataset = datasets.ImageFolder(os.path.join(real_data_dir, 'val'), transform=transform)

    test_loader = DataLoader(test_dataset,
                             batch_size=64,
                             shuffle=False,
                             num_workers=2)

    print(f"--- TRYB PEŁNY: Zbiór treningowy liczy {len(full_train_dataset)} obrazów ---")

    # EKSTRAKCJA CECH (Dla wszystkich obrazów w zbiorze)
    print("\n--- KROK 1: Ekstrakcja cech dla całego zbioru (Dino v2) ---")
    
    # code to get embedings from dino for sampling purposes
    # dataset_dino = datasets.ImageFolder(os.path.join(real_data_dir,"train"), transform=DinoTransform())
    # data_tensor = get_dino_embedings(dataset_dino)
    
    data_tensor = get_resnet50_embedings(full_train_dataset).float()

   
    print("\n--- KROK 2: Ranking i Sampling (Wybór 1000 najlepszych obrazów) ---")
    clusters = hkmg.hierarchical_kmeans_with_resampling(
        data=data_tensor,                                                                           
        n_clusters=[120, 50],
        n_levels=2,
        sample_sizes=[15, 2],
        verbose=False,
    )

    cl = HierarchicalCluster.from_dict(clusters)

    target_subset_size = 8000
    sampled_indices = hs.hierarchical_sampling(cl, target_size=target_subset_size)

    sampled_dataset = Subset(full_train_dataset, sampled_indices)
    print(f"Wyselekcjonowano {len(sampled_indices)} obrazów za pomocą Twojej metody.")

    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model = model.to(device)

    num_epochs = 10
    acc_sampled = train_model(
        model,
        DataLoader(sampled_dataset, batch_size=64, shuffle=True, num_workers=2, persistent_workers=True),
        test_loader,
        device,
        f"RANKING SUBSET ({target_subset_size} images)",
        epochs=num_epochs
    )


    acc_full = train_model(
        model,
        DataLoader(full_train_dataset, batch_size=64, shuffle=True, num_workers=2),
        test_loader,
        device,
        f"FULL DATASET ({len(full_train_dataset)} images)",
        epochs=num_epochs
    )

    print("\n" + "=" * 50)
    print(f"FINALNE PODSUMOWANIE PO {num_epochs} EPOKACH:")
    print(f"Accuracy - Pełny zbiór: {acc_full:.2f}%")
    print(f"Accuracy - Twoja metoda (tylko {target_subset_size} zdjęć): {acc_sampled:.2f}%")

    efektywnosc = acc_sampled / acc_full * 100
    print(f"Twoja metoda osiągnęła {efektywnosc:.1f}% jakości pełnego zbioru, "
          f"używając jedynie {target_subset_size / len(full_train_dataset) * 100:.1f}% danych!")
    print("=" * 50)


if __name__ == "__main__":
    main()


Praca na urządzeniu: cuda
--- TRYB PEŁNY: Zbiór treningowy liczy 9469 obrazów ---

--- KROK 1: Ekstrakcja cech dla całego zbioru (Dino v2) ---


Ekstrakcja cech: 100%|██████████| 74/74 [00:32<00:00,  2.25it/s]



--- KROK 2: Ranking i Sampling (Wybór 1000 najlepszych obrazów) ---
Hierarchical sampling from clusters: 100%|██████████| 50/50 [00:00<00:00, 24574.08it/s]
Wyselekcjonowano 8000 obrazów za pomocą Twojej metody.

[TRENING] Start: RANKING SUBSET (8000 images)


Epoka 10/10: 100%|██████████| 125/125 [00:24<00:00,  5.07it/s]


[WYNIK] RANKING SUBSET (8000 images) Accuracy: 92.82%

[TRENING] Start: FULL DATASET (9469 images)


Epoka 10/10: 100%|██████████| 148/148 [00:35<00:00,  4.12it/s]


[WYNIK] FULL DATASET (9469 images) Accuracy: 91.52%

FINALNE PODSUMOWANIE PO 10 EPOKACH:
Accuracy - Pełny zbiór: 91.52%
Accuracy - Twoja metoda (tylko 8000 zdjęć): 92.82%
Twoja metoda osiągnęła 101.4% jakości pełnego zbioru, używając jedynie 84.5% danych!
